# Testing DRS

1. **Prerequisites:** This script assumes you have basic understanding of the following:
   - Python .
   - What FHIR resources look like. As a start, you could take a look at: [FHIR ResearchStudy Resource](https://hl7.org/fhir/researchstudy.html).
   - GA4GH standard DRS.  Please see:  todo.
   - The Speciman and DocumentReference Resources in dbGaP described here: [dbGaP pilot FHIR Resources](../dbGaP-FHIR-Resource-Intro.md)
  
2. The script will do the following:
   - query a specimen record for a patient in the UDN study.
   - using the specimen, find DocumentReference for the specimen.
   - For each document, find the title and the DRS URI for the document.
   - Fing the access method for the DRS.
   - download the document.
  
2. How to find a few patient in the UDN study using FHIR query?
   - First let's look at a patient resource by using: https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Patient?_count=1
   - Notice the "tag" element telling us that code is the study accession when the CodeSystem is the DbGaPConcept-StudyAccessionNoVersion.
     >  "fullUrl": "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Patient/2545076",
    "resource": {
      "resourceType": "Patient",
      "id": "2545076",
      "meta": {
        "versionId": "1",
        "lastUpdated": "2024-08-08T10:03:42.470-04:00",
        "source": "#vDQIGjX6a8wKoeZd",
        "tag": [ {
          "system": "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/DbGaPConcept-StudyAccessionNoVersion",
          "code": "phs001289"
        } ]
      },
     - using the system and code in the tag element, we can formuate the FHIR query to give me a few patients in a given study:
     - https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Patient?_tag=https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/DbGaPConcept-StudyAccessionNoVersion|phs001232&_count=10
     - Try this on the browser window, it works!
     - Next we will code this in the script.


In [1]:
# Notebook02-DRS-Script-1
# https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/DocumentReference?patient=Patient/3510883

import requests
import json

# Initialize a session and update headers
session = requests.Session()
session.headers.update({
    'Accept': 'application/fhir+json',
    'Content-Type': 'application/x-www-form-urlencoded',
})

# Set up the FHIR server URL and query
dbgap_fhir_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/"
my_query = "DocumentReference?patient=Patient/3510883"

my_fhir_url = dbgap_fhir_url + my_query

# Print the full URL being used for the query
print(f"my_fhir_url is {my_fhir_url}")

# Send the GET request to the FHIR server
response = session.get(my_fhir_url)

# Check if the request was successful
if response.status_code == 200:
    response_json = response.json()
    
    # Print the JSON response in a readable format
    print(json.dumps(response_json, indent=4))
else:
    print(f"Failed to retrieve data: {response.status_code}")

my_fhir_url is https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/DocumentReference?patient=Patient/3510883
{
    "resourceType": "Bundle",
    "id": "0caa6862-b460-446e-8342-334ed6d13cf9",
    "meta": {
        "lastUpdated": "2024-09-30T13:21:30.599-04:00"
    },
    "type": "searchset",
    "total": 3,
    "link": [
        {
            "relation": "self",
            "url": "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/DocumentReference?patient=Patient%2F3510883"
        }
    ],
    "entry": [
        {
            "fullUrl": "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/DocumentReference/fdc1390a6ea757cf06e4ed94fe39f6ce",
            "resource": {
                "resourceType": "DocumentReference",
                "id": "fdc1390a6ea757cf06e4ed94fe39f6ce",
                "meta": {
                    "versionId": "4",
                    "lastUpdated": "2024-09-23T15:13:56.701-04:00",
                    "source": "#fB17Oj3KIr1XZ6kw"
                },
       

In [ ]:
## Look at the content of the DocumentReference records for a patient:
### Notice the "drs://ncbidrs:fdc1390a6ea757cf06e4ed94fe39f6ce" above. fdc1390a6ea757cf06e4ed94fe39f6ce is the DRS id.
# Plug the drs id in this URL: https://locate.be-md.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/<drs id>, so we have:
# https://locate.be-md.ncbi.nlm.nih.gov/ga4gh/drs/v1/objects/fdc1390a6ea757cf06e4ed94fe39f6ce


In [ ]:
{
  "access_methods": [
    {
      "access_id": "4e25559f8c5f79d364447d379cef44a34e846132f6d1d4b758d4e7ecc84cb350",
      "region": "s3.us-east-1",
      "type": "https"
    }
  ],
  "checksums": [
    {
      "checksum": "fdc1390a6ea757cf06e4ed94fe39f6ce",
      "type": "md5"
    }
  ],
  "created_time": "2012-12-22T14:53:09Z",
  "id": "fdc1390a6ea757cf06e4ed94fe39f6ce",
  "name": "ERR211013",
  "self_uri": "drs://locate.be-md.ncbi.nlm.nih.gov/fdc1390a6ea757cf06e4ed94fe39f6ce",
  "size": 222474561
}